# Healthcare Data Engineering Project - Group 5

## Project Objective
Build a complete Healthcare Data Engineering Pipeline that:
- Extracts healthcare data from OLTP sources
- Processes it using PySpark in Databricks
- Implements Bronze → Silver → Gold Medallion Architecture
- Creates Star Schema for healthcare analytics

## Architecture Flow
```
Sample Healthcare Data (OLTP)
        ↓
Bronze Layer (Raw)
        ↓
Data Quality Checks
        ↓
Silver Layer (Clean)
        ↓
Business Transformations
        ↓
Gold Layer (Star Schema)
        ↓
Healthcare Analytics & KPIs
```

## Catalog Structure
- **Catalog**: `healthcare`
- **Schema**: `hospital`
- **Layers**: bronze, silver, gold

In [0]:
%sql
-- Create Healthcare Catalog and Schema
CREATE CATALOG IF NOT EXISTS healthcare;
USE CATALOG healthcare;

CREATE SCHEMA IF NOT EXISTS hospital
COMMENT 'Healthcare hospital data - Bronze, Silver, Gold layers';

USE SCHEMA hospital;

SELECT current_catalog(), current_schema();

current_catalog(),current_schema()
healthcare,hospital


In [0]:
# Generate Sample Healthcare Data - Departments
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from datetime import datetime, timedelta
import random

# Department data
departments_data = [
    (1, 'Cardiology', 'Heart and cardiovascular care', 'Building A, Floor 3'),
    (2, 'Neurology', 'Brain and nervous system treatment', 'Building B, Floor 2'),
    (3, 'Orthopedics', 'Bone and joint care', 'Building A, Floor 1'),
    (4, 'Pediatrics', 'Child healthcare services', 'Building C, Floor 1'),
    (5, 'General Medicine', 'General health consultation', 'Building A, Floor 2'),
    (6, 'Emergency', 'Emergency and critical care', 'Building D, Floor 1'),
    (7, 'Oncology', 'Cancer treatment and care', 'Building B, Floor 3'),
    (8, 'Radiology', 'Imaging and diagnostics', 'Building D, Floor 2'),
    (9, 'Dermatology', 'Skin and cosmetic care', 'Building C, Floor 2'),
    (10, 'Psychiatry', 'Mental health services', 'Building B, Floor 1')
]

departments_df = spark.createDataFrame(
    departments_data,
    ['department_id', 'department_name', 'description', 'location']
)

print(f"Generated {departments_df.count()} departments")
display(departments_df)

Generated 10 departments


department_id,department_name,description,location
1,Cardiology,Heart and cardiovascular care,"Building A, Floor 3"
2,Neurology,Brain and nervous system treatment,"Building B, Floor 2"
3,Orthopedics,Bone and joint care,"Building A, Floor 1"
4,Pediatrics,Child healthcare services,"Building C, Floor 1"
5,General Medicine,General health consultation,"Building A, Floor 2"
6,Emergency,Emergency and critical care,"Building D, Floor 1"
7,Oncology,Cancer treatment and care,"Building B, Floor 3"
8,Radiology,Imaging and diagnostics,"Building D, Floor 2"
9,Dermatology,Skin and cosmetic care,"Building C, Floor 2"
10,Psychiatry,Mental health services,"Building B, Floor 1"


In [0]:
# Generate Sample Healthcare Data - Doctors
doctors_data = []
doctor_names = [
    ('Dr. Sarah', 'Johnson'), ('Dr. Michael', 'Chen'), ('Dr. Emily', 'Williams'),
    ('Dr. James', 'Brown'), ('Dr. Lisa', 'Davis'), ('Dr. Robert', 'Miller'),
    ('Dr. Jennifer', 'Wilson'), ('Dr. David', 'Moore'), ('Dr. Maria', 'Garcia'),
    ('Dr. John', 'Martinez'), ('Dr. Anna', 'Rodriguez'), ('Dr. Chris', 'Lee'),
    ('Dr. Patricia', 'Anderson'), ('Dr. Daniel', 'Taylor'), ('Dr. Linda', 'Thomas'),
    ('Dr. Mark', 'Jackson'), ('Dr. Susan', 'White'), ('Dr. Paul', 'Harris'),
    ('Dr. Karen', 'Martin'), ('Dr. Steven', 'Thompson')
]

specializations = [
    'Cardiologist', 'Neurologist', 'Orthopedic Surgeon', 'Pediatrician',
    'General Physician', 'Emergency Medicine', 'Oncologist', 'Radiologist',
    'Dermatologist', 'Psychiatrist'
]

for i, (first_name, last_name) in enumerate(doctor_names, 1):
    dept_id = ((i - 1) % 10) + 1  # Distribute doctors across departments
    spec = specializations[(i - 1) % 10]
    phone = f"+1-555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
    email = f"{first_name.lower().replace('dr. ', '')}.{last_name.lower()}@hospital.com"
    license = f"MD{random.randint(100000, 999999)}"
    
    doctors_data.append((
        i, first_name, last_name, spec, dept_id, 
        phone, email, license, random.randint(5, 25)
    ))

doctors_df = spark.createDataFrame(
    doctors_data,
    ['doctor_id', 'first_name', 'last_name', 'specialization', 'department_id',
     'phone', 'email', 'license_number', 'years_of_experience']
)

print(f"Generated {doctors_df.count()} doctors")
display(doctors_df)

Generated 20 doctors


doctor_id,first_name,last_name,specialization,department_id,phone,email,license_number,years_of_experience
1,Dr. Sarah,Johnson,Cardiologist,1,+1-555-632-4662,sarah.johnson@hospital.com,MD295248,23
2,Dr. Michael,Chen,Neurologist,2,+1-555-868-6361,michael.chen@hospital.com,MD340233,21
3,Dr. Emily,Williams,Orthopedic Surgeon,3,+1-555-958-4658,emily.williams@hospital.com,MD213840,14
4,Dr. James,Brown,Pediatrician,4,+1-555-859-2533,james.brown@hospital.com,MD925444,8
5,Dr. Lisa,Davis,General Physician,5,+1-555-518-5399,lisa.davis@hospital.com,MD972210,23
6,Dr. Robert,Miller,Emergency Medicine,6,+1-555-749-4839,robert.miller@hospital.com,MD237764,13
7,Dr. Jennifer,Wilson,Oncologist,7,+1-555-522-2256,jennifer.wilson@hospital.com,MD482855,24
8,Dr. David,Moore,Radiologist,8,+1-555-764-4726,david.moore@hospital.com,MD137987,22
9,Dr. Maria,Garcia,Dermatologist,9,+1-555-276-1503,maria.garcia@hospital.com,MD356450,25
10,Dr. John,Martinez,Psychiatrist,10,+1-555-988-2359,john.martinez@hospital.com,MD877531,18


In [0]:
# Generate Sample Healthcare Data - Patients
import random
from datetime import datetime, timedelta

first_names = ['John', 'Jane', 'Michael', 'Sarah', 'David', 'Emily', 'James', 'Maria', 
               'Robert', 'Linda', 'William', 'Patricia', 'Richard', 'Jennifer', 'Thomas']
last_names = ['Smith', 'Johnson', 'Williams', 'Brown', 'Jones', 'Garcia', 'Miller', 
              'Davis', 'Rodriguez', 'Martinez', 'Wilson', 'Anderson', 'Taylor', 'Thomas']

patients_data = []
base_date = datetime(2020, 1, 1)

for i in range(1, 501):  # Generate 500 patients
    first_name = random.choice(first_names)
    last_name = random.choice(last_names)
    gender = random.choice(['M', 'F'])
    
    # Generate DOB (age between 1 and 85)
    age = random.randint(1, 85)
    dob = datetime.now() - timedelta(days=age*365 + random.randint(0, 365))
    
    # Registration date between 2020 and 2026
    days_offset = random.randint(0, 2300)
    reg_date = base_date + timedelta(days=days_offset)
    
    phone = f"+1-555-{random.randint(100, 999)}-{random.randint(1000, 9999)}"
    email = f"{first_name.lower()}.{last_name.lower()}{i}@email.com"
    
    address = f"{random.randint(100, 9999)} {random.choice(['Main', 'Oak', 'Maple', 'Pine'])} St"
    city = random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'])
    state = random.choice(['NY', 'CA', 'IL', 'TX', 'AZ'])
    zip_code = f"{random.randint(10000, 99999)}"
    
    blood_group = random.choice(['A+', 'A-', 'B+', 'B-', 'AB+', 'AB-', 'O+', 'O-'])
    
    patients_data.append((
        i, first_name, last_name, gender, dob, phone, email,
        address, city, state, zip_code, blood_group, reg_date
    ))

patients_df = spark.createDataFrame(
    patients_data,
    ['patient_id', 'first_name', 'last_name', 'gender', 'date_of_birth', 'phone', 'email',
     'address', 'city', 'state', 'zip_code', 'blood_group', 'registration_date']
)

print(f"Generated {patients_df.count()} patients")
display(patients_df.limit(10))

Generated 500 patients


patient_id,first_name,last_name,gender,date_of_birth,phone,email,address,city,state,zip_code,blood_group,registration_date
1,Richard,Miller,F,1991-09-19T12:05:21.844Z,+1-555-355-8929,richard.miller1@email.com,2721 Maple St,Phoenix,NY,87540,AB+,2023-07-30T00:00:00.000Z
2,John,Taylor,M,2015-09-15T12:05:21.846Z,+1-555-578-2917,john.taylor2@email.com,7236 Main St,Los Angeles,NY,73023,AB-,2021-11-23T00:00:00.000Z
3,Linda,Wilson,F,1970-02-05T12:05:21.846Z,+1-555-110-7311,linda.wilson3@email.com,1040 Main St,Los Angeles,AZ,26342,A-,2022-08-04T00:00:00.000Z
4,Linda,Wilson,M,1992-03-09T12:05:21.846Z,+1-555-985-1008,linda.wilson4@email.com,8481 Maple St,New York,AZ,40300,O+,2022-12-01T00:00:00.000Z
5,Linda,Johnson,M,1946-07-18T12:05:21.846Z,+1-555-983-5630,linda.johnson5@email.com,7967 Maple St,New York,AZ,19290,O+,2020-09-27T00:00:00.000Z
6,John,Miller,M,1987-07-21T12:05:21.846Z,+1-555-351-2889,john.miller6@email.com,7151 Maple St,Phoenix,CA,24956,O+,2020-02-08T00:00:00.000Z
7,William,Martinez,F,1979-07-18T12:05:21.846Z,+1-555-408-6638,william.martinez7@email.com,3332 Maple St,Chicago,TX,74356,B-,2023-10-23T00:00:00.000Z
8,Emily,Miller,F,1966-06-11T12:05:21.846Z,+1-555-903-1611,emily.miller8@email.com,3492 Oak St,Phoenix,IL,30519,A-,2022-11-16T00:00:00.000Z
9,Robert,Williams,F,1994-08-11T12:05:21.846Z,+1-555-109-6313,robert.williams9@email.com,4863 Pine St,Los Angeles,IL,77022,AB+,2022-01-01T00:00:00.000Z
10,Sarah,Thomas,M,1985-12-09T12:05:21.846Z,+1-555-705-3496,sarah.thomas10@email.com,650 Maple St,Chicago,TX,63515,AB-,2023-12-28T00:00:00.000Z


In [0]:
# Generate Sample Healthcare Data - Appointments
appointments_data = []
appointment_statuses = ['Scheduled', 'Completed', 'Cancelled', 'No-Show']
appointment_types = ['Consultation', 'Follow-up', 'Emergency', 'Routine Check-up']

base_date = datetime(2024, 1, 1)

for i in range(1, 2001):  # Generate 2000 appointments
    patient_id = random.randint(1, 500)
    doctor_id = random.randint(1, 20)
    
    # Appointment date between Jan 2024 and Aug 2026
    days_offset = random.randint(0, 970)
    appt_date = base_date + timedelta(days=days_offset)
    
    # Appointment time
    hour = random.randint(8, 17)  # 8 AM to 5 PM
    minute = random.choice([0, 15, 30, 45])
    appt_time = appt_date.replace(hour=hour, minute=minute)
    
    # Duration: 15, 30, 45, or 60 minutes
    duration_minutes = random.choice([15, 30, 45, 60])
    end_time = appt_time + timedelta(minutes=duration_minutes)
    
    status = random.choice(appointment_statuses)
    appt_type = random.choice(appointment_types)
    reason = random.choice(['Chest pain', 'Fever', 'Headache', 'Back pain', 'Routine checkup',
                           'Follow-up visit', 'Skin rash', 'Joint pain', 'Annual physical'])
    
    appointments_data.append((
        i, patient_id, doctor_id, appt_time, end_time, 
        duration_minutes, status, appt_type, reason
    ))

appointments_df = spark.createDataFrame(
    appointments_data,
    ['appointment_id', 'patient_id', 'doctor_id', 'appointment_date', 'end_time',
     'duration_minutes', 'status', 'appointment_type', 'reason_for_visit']
)

print(f"Generated {appointments_df.count()} appointments")
display(appointments_df.limit(10))

Generated 2000 appointments


appointment_id,patient_id,doctor_id,appointment_date,end_time,duration_minutes,status,appointment_type,reason_for_visit
1,338,15,2024-07-05T16:00:00.000Z,2024-07-05T16:15:00.000Z,15,No-Show,Consultation,Annual physical
2,162,9,2024-07-25T17:45:00.000Z,2024-07-25T18:30:00.000Z,45,Completed,Routine Check-up,Back pain
3,484,14,2026-04-26T09:00:00.000Z,2026-04-26T10:00:00.000Z,60,Scheduled,Consultation,Annual physical
4,414,18,2024-10-31T14:45:00.000Z,2024-10-31T15:15:00.000Z,30,Completed,Follow-up,Chest pain
5,292,18,2024-12-11T17:45:00.000Z,2024-12-11T18:45:00.000Z,60,Scheduled,Emergency,Joint pain
6,354,18,2024-01-04T15:45:00.000Z,2024-01-04T16:00:00.000Z,15,Completed,Emergency,Skin rash
7,416,19,2024-10-13T15:00:00.000Z,2024-10-13T16:00:00.000Z,60,No-Show,Consultation,Joint pain
8,284,18,2025-10-27T09:30:00.000Z,2025-10-27T10:30:00.000Z,60,Scheduled,Routine Check-up,Follow-up visit
9,43,3,2024-10-10T14:45:00.000Z,2024-10-10T15:00:00.000Z,15,Completed,Consultation,Fever
10,150,10,2025-11-25T10:15:00.000Z,2025-11-25T10:30:00.000Z,15,No-Show,Routine Check-up,Fever


In [0]:
# Generate Sample Healthcare Data - Diagnoses
diagnoses_data = [
    (1, 'Hypertension', 'High blood pressure', 'Chronic'),
    (2, 'Type 2 Diabetes', 'Diabetes mellitus', 'Chronic'),
    (3, 'Asthma', 'Respiratory condition', 'Chronic'),
    (4, 'Migraine', 'Severe headache', 'Episodic'),
    (5, 'Pneumonia', 'Lung infection', 'Acute'),
    (6, 'Bronchitis', 'Airway inflammation', 'Acute'),
    (7, 'Arthritis', 'Joint inflammation', 'Chronic'),
    (8, 'Coronary Artery Disease', 'Heart disease', 'Chronic'),
    (9, 'Depression', 'Mental health disorder', 'Chronic'),
    (10, 'Anxiety Disorder', 'Mental health condition', 'Chronic'),
    (11, 'Influenza', 'Flu virus', 'Acute'),
    (12, 'Gastritis', 'Stomach inflammation', 'Acute'),
    (13, 'Fracture', 'Broken bone', 'Acute'),
    (14, 'Skin Infection', 'Bacterial skin infection', 'Acute'),
    (15, 'Back Pain', 'Lower back pain', 'Chronic')
]

diagnoses_df = spark.createDataFrame(
    diagnoses_data,
    ['diagnosis_id', 'diagnosis_name', 'description', 'category']
)

# Generate Treatments
treatments_data = [
    (1, 'Medication Therapy', 'Prescription medication', 150.00),
    (2, 'Physical Therapy', 'Rehabilitation exercises', 200.00),
    (3, 'Surgery', 'Surgical procedure', 5000.00),
    (4, 'Counseling', 'Mental health counseling', 180.00),
    (5, 'X-Ray', 'Radiological imaging', 300.00),
    (6, 'MRI Scan', 'Magnetic resonance imaging', 1200.00),
    (7, 'Blood Test', 'Laboratory blood work', 75.00),
    (8, 'CT Scan', 'Computed tomography', 800.00),
    (9, 'Ultrasound', 'Ultrasound imaging', 350.00),
    (10, 'Injection', 'Therapeutic injection', 120.00),
    (11, 'Chemotherapy', 'Cancer treatment', 3500.00),
    (12, 'Dialysis', 'Kidney treatment', 2000.00),
    (13, 'ECG', 'Electrocardiogram', 150.00),
    (14, 'Vaccination', 'Immunization', 50.00),
    (15, 'Cast Application', 'Fracture treatment', 250.00)
]

treatments_df = spark.createDataFrame(
    treatments_data,
    ['treatment_id', 'treatment_name', 'description', 'base_cost']
)

print(f"Generated {diagnoses_df.count()} diagnoses and {treatments_df.count()} treatments")
display(diagnoses_df)
display(treatments_df)

Generated 15 diagnoses and 15 treatments


diagnosis_id,diagnosis_name,description,category
1,Hypertension,High blood pressure,Chronic
2,Type 2 Diabetes,Diabetes mellitus,Chronic
3,Asthma,Respiratory condition,Chronic
4,Migraine,Severe headache,Episodic
5,Pneumonia,Lung infection,Acute
6,Bronchitis,Airway inflammation,Acute
7,Arthritis,Joint inflammation,Chronic
8,Coronary Artery Disease,Heart disease,Chronic
9,Depression,Mental health disorder,Chronic
10,Anxiety Disorder,Mental health condition,Chronic


treatment_id,treatment_name,description,base_cost
1,Medication Therapy,Prescription medication,150.0
2,Physical Therapy,Rehabilitation exercises,200.0
3,Surgery,Surgical procedure,5000.0
4,Counseling,Mental health counseling,180.0
5,X-Ray,Radiological imaging,300.0
6,MRI Scan,Magnetic resonance imaging,1200.0
7,Blood Test,Laboratory blood work,75.0
8,CT Scan,Computed tomography,800.0
9,Ultrasound,Ultrasound imaging,350.0
10,Injection,Therapeutic injection,120.0


In [0]:
# Generate Sample Healthcare Data - Patient Treatments (linking patients to treatments)
patient_treatments_data = []

for i in range(1, 1501):  # Generate 1500 treatment records
    patient_id = random.randint(1, 500)
    appointment_id = random.randint(1, 2000)
    treatment_id = random.randint(1, 15)
    diagnosis_id = random.randint(1, 15)
    
    # Treatment date
    days_offset = random.randint(0, 970)
    treatment_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    # Get base cost and add variation
    base_cost = [t[3] for t in treatments_data if t[0] == treatment_id][0]
    import builtins
    cost = builtins.round(base_cost * random.uniform(0.9, 1.3), 2)
    
    notes = random.choice([
        'Treatment completed successfully',
        'Patient responding well',
        'Follow-up required',
        'Initial treatment phase',
        'Continued care needed'
    ])
    
    patient_treatments_data.append((
        i, patient_id, appointment_id, treatment_id, diagnosis_id,
        treatment_date, cost, notes
    ))

patient_treatments_df = spark.createDataFrame(
    patient_treatments_data,
    ['treatment_record_id', 'patient_id', 'appointment_id', 'treatment_id', 
     'diagnosis_id', 'treatment_date', 'cost', 'notes']
)

print(f"Generated {patient_treatments_df.count()} patient treatment records")
display(patient_treatments_df.limit(10))

Generated 1500 patient treatment records


treatment_record_id,patient_id,appointment_id,treatment_id,diagnosis_id,treatment_date,cost,notes
1,475,949,15,9,2025-07-26T00:00:00.000Z,308.88,Patient responding well
2,286,1098,4,1,2024-11-21T00:00:00.000Z,201.14,Continued care needed
3,366,100,8,3,2025-12-12T00:00:00.000Z,916.76,Continued care needed
4,316,510,9,14,2025-03-15T00:00:00.000Z,425.65,Follow-up required
5,463,1036,6,8,2025-12-06T00:00:00.000Z,1475.37,Initial treatment phase
6,287,648,10,9,2024-01-17T00:00:00.000Z,132.5,Follow-up required
7,150,454,8,9,2024-10-16T00:00:00.000Z,929.06,Initial treatment phase
8,154,1649,12,15,2024-05-30T00:00:00.000Z,1915.9,Follow-up required
9,35,905,15,12,2025-04-23T00:00:00.000Z,231.75,Continued care needed
10,243,1060,10,11,2024-12-11T00:00:00.000Z,139.83,Continued care needed


In [0]:
# Generate Sample Healthcare Data - Medications
medications_data = [
    (1, 'Lisinopril', 'ACE inhibitor for hypertension', 'Tablet', 25.50),
    (2, 'Metformin', 'Diabetes medication', 'Tablet', 18.75),
    (3, 'Albuterol', 'Asthma inhaler', 'Inhaler', 45.00),
    (4, 'Ibuprofen', 'Pain reliever', 'Tablet', 12.00),
    (5, 'Amoxicillin', 'Antibiotic', 'Capsule', 22.50),
    (6, 'Atorvastatin', 'Cholesterol medication', 'Tablet', 35.00),
    (7, 'Omeprazole', 'Acid reflux medication', 'Capsule', 28.00),
    (8, 'Sertraline', 'Antidepressant', 'Tablet', 42.00),
    (9, 'Levothyroxine', 'Thyroid medication', 'Tablet', 15.00),
    (10, 'Amlodipine', 'Blood pressure medication', 'Tablet', 20.00),
    (11, 'Gabapentin', 'Nerve pain medication', 'Capsule', 38.00),
    (12, 'Prednisone', 'Corticosteroid', 'Tablet', 16.50),
    (13, 'Azithromycin', 'Antibiotic', 'Tablet', 32.00),
    (14, 'Hydrochlorothiazide', 'Diuretic', 'Tablet', 14.00),
    (15, 'Furosemide', 'Diuretic', 'Tablet', 18.00)
]

medications_df = spark.createDataFrame(
    medications_data,
    ['medication_id', 'medication_name', 'description', 'dosage_form', 'unit_price']
)

# Generate Prescriptions
prescriptions_data = []

for i in range(1, 1201):  # Generate 1200 prescriptions
    patient_id = random.randint(1, 500)
    doctor_id = random.randint(1, 20)
    medication_id = random.randint(1, 15)
    
    days_offset = random.randint(0, 970)
    prescription_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    dosage = random.choice(['10mg', '20mg', '50mg', '100mg', '250mg', '500mg'])
    frequency = random.choice(['Once daily', 'Twice daily', 'Three times daily', 'As needed'])
    duration_days = random.choice([7, 14, 30, 60, 90])
    quantity = random.randint(10, 90)
    refills = random.randint(0, 5)
    
    prescriptions_data.append((
        i, patient_id, doctor_id, medication_id, prescription_date,
        dosage, frequency, duration_days, quantity, refills
    ))

prescriptions_df = spark.createDataFrame(
    prescriptions_data,
    ['prescription_id', 'patient_id', 'doctor_id', 'medication_id', 'prescription_date',
     'dosage', 'frequency', 'duration_days', 'quantity', 'refills_allowed']
)

print(f"Generated {medications_df.count()} medications and {prescriptions_df.count()} prescriptions")
display(medications_df)
display(prescriptions_df.limit(10))

Generated 15 medications and 1200 prescriptions


medication_id,medication_name,description,dosage_form,unit_price
1,Lisinopril,ACE inhibitor for hypertension,Tablet,25.5
2,Metformin,Diabetes medication,Tablet,18.75
3,Albuterol,Asthma inhaler,Inhaler,45.0
4,Ibuprofen,Pain reliever,Tablet,12.0
5,Amoxicillin,Antibiotic,Capsule,22.5
6,Atorvastatin,Cholesterol medication,Tablet,35.0
7,Omeprazole,Acid reflux medication,Capsule,28.0
8,Sertraline,Antidepressant,Tablet,42.0
9,Levothyroxine,Thyroid medication,Tablet,15.0
10,Amlodipine,Blood pressure medication,Tablet,20.0


prescription_id,patient_id,doctor_id,medication_id,prescription_date,dosage,frequency,duration_days,quantity,refills_allowed
1,96,1,9,2024-02-06T00:00:00.000Z,500mg,Once daily,30,31,3
2,287,20,9,2025-04-23T00:00:00.000Z,50mg,Once daily,7,70,3
3,171,2,3,2025-07-22T00:00:00.000Z,20mg,As needed,90,52,5
4,356,14,1,2025-08-15T00:00:00.000Z,50mg,Twice daily,60,18,1
5,134,6,5,2025-11-02T00:00:00.000Z,50mg,Once daily,30,10,0
6,182,16,1,2026-06-04T00:00:00.000Z,500mg,Three times daily,90,43,0
7,337,7,1,2024-04-28T00:00:00.000Z,250mg,Three times daily,14,11,0
8,46,11,4,2025-07-06T00:00:00.000Z,250mg,As needed,30,49,1
9,194,9,13,2025-05-25T00:00:00.000Z,250mg,As needed,14,33,4
10,268,7,3,2025-12-27T00:00:00.000Z,100mg,Three times daily,30,34,5


In [0]:
# Generate Sample Healthcare Data - Lab Tests
lab_tests_data = [
    (1, 'Complete Blood Count (CBC)', 'Blood cell counts', 85.00),
    (2, 'Basic Metabolic Panel', 'Blood chemistry', 95.00),
    (3, 'Lipid Panel', 'Cholesterol levels', 75.00),
    (4, 'Hemoglobin A1C', 'Diabetes monitoring', 65.00),
    (5, 'Thyroid Function Test', 'TSH levels', 120.00),
    (6, 'Liver Function Test', 'Liver enzymes', 110.00),
    (7, 'Urinalysis', 'Urine analysis', 45.00),
    (8, 'COVID-19 Test', 'Viral detection', 125.00),
    (9, 'Vitamin D Test', 'Vitamin D levels', 90.00),
    (10, 'PSA Test', 'Prostate screening', 95.00)
]

lab_tests_df = spark.createDataFrame(
    lab_tests_data,
    ['test_id', 'test_name', 'description', 'cost']
)

# Generate Lab Test Results
lab_results_data = []

for i in range(1, 1001):  # Generate 1000 lab test results
    patient_id = random.randint(1, 500)
    test_id = random.randint(1, 10)
    doctor_id = random.randint(1, 20)
    
    days_offset = random.randint(0, 970)
    test_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    # Result date is 1-5 days after test date
    result_date = test_date + timedelta(days=random.randint(1, 5))
    
    result_value = random.choice([
        'Normal', 'Abnormal', 'High', 'Low', 'Positive', 'Negative',
        '150 mg/dL', '95 mg/dL', '6.5%', '180 U/L', '12.5 g/dL'
    ])
    
    status = random.choice(['Completed', 'In Progress', 'Pending'])
    notes = random.choice(['Normal range', 'Requires follow-up', 'Repeat test needed', 'Stable'])
    
    lab_results_data.append((
        i, patient_id, test_id, doctor_id, test_date, result_date,
        result_value, status, notes
    ))

lab_results_df = spark.createDataFrame(
    lab_results_data,
    ['result_id', 'patient_id', 'test_id', 'doctor_id', 'test_date', 'result_date',
     'result_value', 'status', 'notes']
)

print(f"Generated {lab_tests_df.count()} lab tests and {lab_results_df.count()} lab results")
display(lab_tests_df)
display(lab_results_df.limit(10))

Generated 10 lab tests and 1000 lab results


test_id,test_name,description,cost
1,Complete Blood Count (CBC),Blood cell counts,85.0
2,Basic Metabolic Panel,Blood chemistry,95.0
3,Lipid Panel,Cholesterol levels,75.0
4,Hemoglobin A1C,Diabetes monitoring,65.0
5,Thyroid Function Test,TSH levels,120.0
6,Liver Function Test,Liver enzymes,110.0
7,Urinalysis,Urine analysis,45.0
8,COVID-19 Test,Viral detection,125.0
9,Vitamin D Test,Vitamin D levels,90.0
10,PSA Test,Prostate screening,95.0


result_id,patient_id,test_id,doctor_id,test_date,result_date,result_value,status,notes
1,287,2,19,2025-08-19T00:00:00.000Z,2025-08-21T00:00:00.000Z,High,Pending,Repeat test needed
2,271,1,7,2025-11-22T00:00:00.000Z,2025-11-27T00:00:00.000Z,Normal,Pending,Normal range
3,261,3,1,2025-03-03T00:00:00.000Z,2025-03-07T00:00:00.000Z,Positive,In Progress,Requires follow-up
4,330,2,9,2025-03-24T00:00:00.000Z,2025-03-28T00:00:00.000Z,Positive,Completed,Stable
5,321,6,16,2025-02-08T00:00:00.000Z,2025-02-10T00:00:00.000Z,95 mg/dL,Completed,Normal range
6,211,10,3,2025-09-18T00:00:00.000Z,2025-09-20T00:00:00.000Z,95 mg/dL,In Progress,Repeat test needed
7,163,9,2,2024-08-03T00:00:00.000Z,2024-08-04T00:00:00.000Z,95 mg/dL,Completed,Requires follow-up
8,8,6,7,2024-08-02T00:00:00.000Z,2024-08-07T00:00:00.000Z,Abnormal,Completed,Stable
9,391,6,17,2026-04-04T00:00:00.000Z,2026-04-09T00:00:00.000Z,95 mg/dL,Pending,Stable
10,481,4,2,2024-10-23T00:00:00.000Z,2024-10-28T00:00:00.000Z,Low,Completed,Requires follow-up


In [0]:
# Generate Sample Healthcare Data - Insurance Providers
insurance_providers_data = [
    (1, 'Blue Cross Blue Shield', '+1-800-555-1234', 'info@bcbs.com', '75%'),
    (2, 'United Healthcare', '+1-800-555-2345', 'support@uhc.com', '80%'),
    (3, 'Aetna', '+1-800-555-3456', 'claims@aetna.com', '70%'),
    (4, 'Cigna', '+1-800-555-4567', 'service@cigna.com', '75%'),
    (5, 'Humana', '+1-800-555-5678', 'help@humana.com', '65%'),
    (6, 'Medicare', '+1-800-555-6789', 'info@medicare.gov', '80%'),
    (7, 'Medicaid', '+1-800-555-7890', 'support@medicaid.gov', '90%'),
    (8, 'Kaiser Permanente', '+1-800-555-8901', 'service@kaiser.com', '85%')
]

insurance_providers_df = spark.createDataFrame(
    insurance_providers_data,
    ['provider_id', 'provider_name', 'phone', 'email', 'coverage_percentage']
)

# Generate Patient Insurance
patient_insurance_data = []

for i in range(1, 451):  # 450 patients have insurance (90% of 500 patients)
    patient_id = i
    provider_id = random.randint(1, 8)
    policy_number = f"POL-{random.randint(100000, 999999)}"
    
    start_date = datetime(2020, 1, 1) + timedelta(days=random.randint(0, 1500))
    end_date = start_date + timedelta(days=random.choice([365, 730, 1095]))  # 1, 2, or 3 years
    
    is_active = 'Yes' if end_date > datetime.now() else 'No'
    
    patient_insurance_data.append((
        i, patient_id, provider_id, policy_number, start_date, end_date, is_active
    ))

patient_insurance_df = spark.createDataFrame(
    patient_insurance_data,
    ['insurance_id', 'patient_id', 'provider_id', 'policy_number', 
     'start_date', 'end_date', 'is_active']
)

print(f"Generated {insurance_providers_df.count()} insurance providers")
print(f"Generated {patient_insurance_df.count()} patient insurance records")
display(insurance_providers_df)
display(patient_insurance_df.limit(10))

Generated 8 insurance providers
Generated 450 patient insurance records


provider_id,provider_name,phone,email,coverage_percentage
1,Blue Cross Blue Shield,+1-800-555-1234,info@bcbs.com,75%
2,United Healthcare,+1-800-555-2345,support@uhc.com,80%
3,Aetna,+1-800-555-3456,claims@aetna.com,70%
4,Cigna,+1-800-555-4567,service@cigna.com,75%
5,Humana,+1-800-555-5678,help@humana.com,65%
6,Medicare,+1-800-555-6789,info@medicare.gov,80%
7,Medicaid,+1-800-555-7890,support@medicaid.gov,90%
8,Kaiser Permanente,+1-800-555-8901,service@kaiser.com,85%


insurance_id,patient_id,provider_id,policy_number,start_date,end_date,is_active
1,1,2,POL-553231,2024-01-23T00:00:00.000Z,2026-01-22T00:00:00.000Z,No
2,2,3,POL-260706,2020-06-04T00:00:00.000Z,2023-06-04T00:00:00.000Z,No
3,3,2,POL-207577,2021-09-14T00:00:00.000Z,2022-09-14T00:00:00.000Z,No
4,4,2,POL-561999,2021-04-23T00:00:00.000Z,2023-04-23T00:00:00.000Z,No
5,5,2,POL-592919,2021-11-08T00:00:00.000Z,2024-11-07T00:00:00.000Z,No
6,6,8,POL-548648,2023-09-06T00:00:00.000Z,2025-09-05T00:00:00.000Z,No
7,7,8,POL-776774,2020-06-14T00:00:00.000Z,2023-06-14T00:00:00.000Z,No
8,8,5,POL-415551,2023-01-08T00:00:00.000Z,2024-01-08T00:00:00.000Z,No
9,9,1,POL-663317,2022-09-20T00:00:00.000Z,2024-09-19T00:00:00.000Z,No
10,10,6,POL-234335,2021-08-24T00:00:00.000Z,2023-08-24T00:00:00.000Z,No


In [0]:
# Generate Sample Healthcare Data - Billing
billing_data = []

for i in range(1, 1801):  # Generate 1800 billing records
    patient_id = random.randint(1, 500)
    appointment_id = random.randint(1, 2000)
    
    days_offset = random.randint(0, 970)
    billing_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    # Total amount (sum of various charges)
    consultation_fee = random.uniform(100, 300)
    treatment_charges = random.uniform(200, 2000)
    lab_charges = random.uniform(50, 500)
    medication_charges = random.uniform(30, 400)
    
    import builtins
    total_amount = builtins.round(consultation_fee + treatment_charges + lab_charges + medication_charges, 2)
    
    # Insurance coverage (if patient has insurance)
    insurance_coverage = 0.0
    if patient_id <= 450:  # Patients with insurance
        coverage_pct = random.uniform(0.65, 0.85)
        insurance_coverage = builtins.round(total_amount * coverage_pct, 2)
    
    patient_responsibility = builtins.round(total_amount - insurance_coverage, 2)
    
    status = random.choice(['Paid', 'Pending', 'Overdue', 'Partially Paid'])
    
    billing_data.append((
        i, patient_id, appointment_id, billing_date, total_amount,
        insurance_coverage, patient_responsibility, status
    ))

billing_df = spark.createDataFrame(
    billing_data,
    ['bill_id', 'patient_id', 'appointment_id', 'billing_date', 'total_amount',
     'insurance_coverage', 'patient_responsibility', 'status']
)

print(f"Generated {billing_df.count()} billing records")
print(f"Total Billed: ${billing_df.agg(sum('total_amount')).collect()[0][0]:,.2f}")
display(billing_df.limit(10))

Generated 1800 billing records
Total Billed: $3,253,834.11


bill_id,patient_id,appointment_id,billing_date,total_amount,insurance_coverage,patient_responsibility,status
1,169,1472,2024-09-29T00:00:00.000Z,1506.69,1125.67,381.02,Pending
2,425,596,2024-12-24T00:00:00.000Z,2087.65,1374.18,713.47,Overdue
3,177,164,2024-03-29T00:00:00.000Z,2587.36,2023.89,563.47,Pending
4,169,1323,2024-10-25T00:00:00.000Z,1525.36,1005.99,519.37,Partially Paid
5,353,540,2024-04-08T00:00:00.000Z,1150.89,906.48,244.41,Partially Paid
6,227,1376,2024-12-12T00:00:00.000Z,1800.84,1196.03,604.81,Paid
7,261,1188,2026-06-19T00:00:00.000Z,878.08,636.49,241.59,Pending
8,12,1792,2025-03-24T00:00:00.000Z,1904.12,1489.47,414.65,Overdue
9,233,1738,2024-07-03T00:00:00.000Z,2828.85,2280.96,547.89,Paid
10,119,380,2025-05-30T00:00:00.000Z,1320.79,1032.99,287.8,Pending


In [0]:
# Generate Sample Healthcare Data - Payments
payments_data = []

for i in range(1, 1501):  # Generate 1500 payment records
    bill_id = random.randint(1, 1800)
    patient_id = random.randint(1, 500)
    
    days_offset = random.randint(0, 970)
    payment_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    # Payment amount (varies based on status)
    import builtins
    payment_amount = builtins.round(random.uniform(50, 1500), 2)
    
    payment_method = random.choice(['Credit Card', 'Debit Card', 'Cash', 'Check', 'Insurance', 'Online Payment'])
    transaction_id = f"TXN-{random.randint(1000000, 9999999)}"
    
    payments_data.append((
        i, bill_id, patient_id, payment_date, payment_amount, payment_method, transaction_id
    ))

payments_df = spark.createDataFrame(
    payments_data,
    ['payment_id', 'bill_id', 'patient_id', 'payment_date', 'payment_amount',
     'payment_method', 'transaction_id']
)

print(f"Generated {payments_df.count()} payment records")
print(f"Total Payments: ${payments_df.agg(sum('payment_amount')).collect()[0][0]:,.2f}")
display(payments_df.limit(10))

Generated 1500 payment records
Total Payments: $1,163,086.11


payment_id,bill_id,patient_id,payment_date,payment_amount,payment_method,transaction_id
1,42,183,2025-11-21T00:00:00.000Z,136.15,Debit Card,TXN-5226730
2,181,118,2024-11-14T00:00:00.000Z,1199.78,Debit Card,TXN-1818753
3,352,228,2024-05-08T00:00:00.000Z,1188.56,Cash,TXN-5651009
4,729,184,2025-11-18T00:00:00.000Z,176.9,Debit Card,TXN-1302681
5,296,50,2026-04-15T00:00:00.000Z,446.1,Online Payment,TXN-9028548
6,451,387,2024-02-14T00:00:00.000Z,774.56,Check,TXN-2448265
7,439,427,2024-04-07T00:00:00.000Z,849.11,Credit Card,TXN-8263105
8,1302,200,2024-02-09T00:00:00.000Z,1093.15,Check,TXN-3196014
9,530,129,2024-05-24T00:00:00.000Z,1445.18,Cash,TXN-4253162
10,1013,469,2024-08-14T00:00:00.000Z,1348.88,Credit Card,TXN-6481157


In [0]:
# Generate Sample Healthcare Data - Insurance Claims
insurance_claims_data = []

for i in range(1, 1301):  # Generate 1300 insurance claims
    bill_id = random.randint(1, 1800)
    patient_id = random.randint(1, 450)  # Only patients with insurance
    provider_id = random.randint(1, 8)
    
    days_offset = random.randint(0, 970)
    claim_date = datetime(2024, 1, 1) + timedelta(days=days_offset)
    
    import builtins
    claim_amount = builtins.round(random.uniform(500, 5000), 2)
    approved_amount = builtins.round(claim_amount * random.uniform(0.7, 1.0), 2)
    
    claim_status = random.choice(['Approved', 'Pending', 'Rejected', 'Under Review'])
    
    if claim_status == 'Approved':
        processed_date = claim_date + timedelta(days=random.randint(5, 30))
    else:
        processed_date = None
    
    claim_number = f"CLM-{random.randint(100000, 999999)}"
    
    insurance_claims_data.append((
        i, bill_id, patient_id, provider_id, claim_date, claim_amount,
        approved_amount, claim_status, processed_date, claim_number
    ))

insurance_claims_df = spark.createDataFrame(
    insurance_claims_data,
    ['claim_id', 'bill_id', 'patient_id', 'provider_id', 'claim_date', 'claim_amount',
     'approved_amount', 'claim_status', 'processed_date', 'claim_number']
)

print(f"Generated {insurance_claims_df.count()} insurance claims")
print(f"Total Claimed: ${insurance_claims_df.agg(sum('claim_amount')).collect()[0][0]:,.2f}")
print(f"Total Approved: ${insurance_claims_df.agg(sum('approved_amount')).collect()[0][0]:,.2f}")
display(insurance_claims_df.limit(10))

Generated 1300 insurance claims
Total Claimed: $3,651,524.86
Total Approved: $3,105,231.39


claim_id,bill_id,patient_id,provider_id,claim_date,claim_amount,approved_amount,claim_status,processed_date,claim_number
1,879,315,3,2026-08-17T00:00:00.000Z,1862.24,1346.16,Rejected,null,CLM-315430
2,1547,173,1,2024-08-15T00:00:00.000Z,3160.13,2249.94,Approved,2024-09-09T00:00:00.000Z,CLM-599538
3,1066,450,7,2024-04-14T00:00:00.000Z,3246.16,2550.31,Under Review,null,CLM-689442
4,1456,48,8,2026-04-05T00:00:00.000Z,1504.91,1412.11,Pending,null,CLM-567383
5,1136,328,2,2026-05-09T00:00:00.000Z,3382.16,3292.57,Under Review,null,CLM-600551
6,705,6,3,2024-09-27T00:00:00.000Z,3536.98,2687.06,Under Review,null,CLM-342303
7,314,133,7,2024-03-29T00:00:00.000Z,709.64,695.91,Approved,2024-04-17T00:00:00.000Z,CLM-395640
8,1195,36,5,2026-03-15T00:00:00.000Z,3204.26,2909.41,Pending,null,CLM-684649
9,1701,434,5,2024-11-05T00:00:00.000Z,2033.55,1750.05,Pending,null,CLM-278828
10,692,208,7,2026-07-09T00:00:00.000Z,2729.72,2561.82,Approved,2026-07-18T00:00:00.000Z,CLM-203563


In [0]:
# BRONZE LAYER - Save raw healthcare data with metadata
from pyspark.sql.functions import current_timestamp, lit

batch_id = "BATCH_001"
source_system = "OLTP_HEALTHCARE_DB"

# Function to add Bronze metadata
def save_to_bronze(df, table_name, record_count):
    bronze_df = df \
        .withColumn("load_timestamp", current_timestamp()) \
        .withColumn("batch_id", lit(batch_id)) \
        .withColumn("source_system", lit(source_system)) \
        .withColumn("record_count", lit(record_count))
    
    bronze_df.write.mode("overwrite").saveAsTable(f"healthcare.hospital.bronze_{table_name}")
    print(f"✓ Saved {record_count} records to bronze_{table_name}")
    return record_count

print("=== BRONZE LAYER: Creating Raw Tables ===")
print()

# Save all tables to Bronze layer
total_records = 0
total_records += save_to_bronze(departments_df, "departments", departments_df.count())
total_records += save_to_bronze(doctors_df, "doctors", doctors_df.count())
total_records += save_to_bronze(patients_df, "patients", patients_df.count())
total_records += save_to_bronze(appointments_df, "appointments", appointments_df.count())
total_records += save_to_bronze(diagnoses_df, "diagnoses", diagnoses_df.count())
total_records += save_to_bronze(treatments_df, "treatments", treatments_df.count())
total_records += save_to_bronze(patient_treatments_df, "patient_treatments", patient_treatments_df.count())
total_records += save_to_bronze(medications_df, "medications", medications_df.count())
total_records += save_to_bronze(prescriptions_df, "prescriptions", prescriptions_df.count())
total_records += save_to_bronze(lab_tests_df, "lab_tests", lab_tests_df.count())
total_records += save_to_bronze(lab_results_df, "lab_results", lab_results_df.count())
total_records += save_to_bronze(insurance_providers_df, "insurance_providers", insurance_providers_df.count())
total_records += save_to_bronze(patient_insurance_df, "patient_insurance", patient_insurance_df.count())
total_records += save_to_bronze(billing_df, "billing", billing_df.count())
total_records += save_to_bronze(payments_df, "payments", payments_df.count())
total_records += save_to_bronze(insurance_claims_df, "insurance_claims", insurance_claims_df.count())

print()
print(f"=== BRONZE LAYER COMPLETE ===")
print(f"Total Records Loaded: {total_records:,}")
print(f"Batch ID: {batch_id}")
print(f"Source System: {source_system}")

=== BRONZE LAYER: Creating Raw Tables ===

✓ Saved 10 records to bronze_departments
✓ Saved 20 records to bronze_doctors
✓ Saved 500 records to bronze_patients
✓ Saved 2000 records to bronze_appointments
✓ Saved 15 records to bronze_diagnoses
✓ Saved 15 records to bronze_treatments
✓ Saved 1500 records to bronze_patient_treatments
✓ Saved 15 records to bronze_medications
✓ Saved 1200 records to bronze_prescriptions
✓ Saved 10 records to bronze_lab_tests
✓ Saved 1000 records to bronze_lab_results
✓ Saved 8 records to bronze_insurance_providers
✓ Saved 450 records to bronze_patient_insurance
✓ Saved 1800 records to bronze_billing
✓ Saved 1500 records to bronze_payments
✓ Saved 1300 records to bronze_insurance_claims

=== BRONZE LAYER COMPLETE ===
Total Records Loaded: 11,343
Batch ID: BATCH_001
Source System: OLTP_HEALTHCARE_DB


In [0]:
# DATA QUALITY CHECKS
from pyspark.sql.functions import col, count, when, isnan, isnull, countDistinct

print("=== DATA QUALITY CHECKS ===")
print()

# Check 1: Duplicate Records
print("1. DUPLICATE CHECKS:")
patients_dups = spark.table("healthcare.hospital.bronze_patients") \
    .groupBy("patient_id").count().filter(col("count") > 1).count()
print(f"   - Duplicate Patient IDs: {patients_dups}")

appointments_dups = spark.table("healthcare.hospital.bronze_appointments") \
    .groupBy("appointment_id").count().filter(col("count") > 1).count()
print(f"   - Duplicate Appointment IDs: {appointments_dups}")

billing_dups = spark.table("healthcare.hospital.bronze_billing") \
    .groupBy("bill_id").count().filter(col("count") > 1).count()
print(f"   - Duplicate Bill IDs: {billing_dups}")
print()

# Check 2: Null Validation
print("2. NULL VALUE CHECKS:")
patients_nulls = spark.table("healthcare.hospital.bronze_patients") \
    .filter(col("patient_id").isNull() | col("first_name").isNull() | col("date_of_birth").isNull()).count()
print(f"   - Patients with null critical fields: {patients_nulls}")

appointments_nulls = spark.table("healthcare.hospital.bronze_appointments") \
    .filter(col("patient_id").isNull() | col("doctor_id").isNull() | col("appointment_date").isNull()).count()
print(f"   - Appointments with null critical fields: {appointments_nulls}")
print()

# Check 3: Date Validation
print("3. DATE LOGIC CHECKS:")
bronze_appts = spark.table("healthcare.hospital.bronze_appointments")
invalid_dates = bronze_appts.filter(col("appointment_date") < col("appointment_date").cast("date")).count()
print(f"   - Invalid appointment dates: {invalid_dates}")
print()

print("=== DATA QUALITY CHECKS COMPLETE ===")

=== DATA QUALITY CHECKS ===

1. DUPLICATE CHECKS:
   - Duplicate Patient IDs: 0
   - Duplicate Appointment IDs: 0
   - Duplicate Bill IDs: 0

2. NULL VALUE CHECKS:
   - Patients with null critical fields: 0
   - Appointments with null critical fields: 0

3. DATE LOGIC CHECKS:
   - Invalid appointment dates: 0

=== DATA QUALITY CHECKS COMPLETE ===


In [0]:
# SILVER LAYER - Clean and standardize healthcare data
from pyspark.sql.functions import trim, upper, lower, regexp_replace, to_date, year, month, dayofmonth

print("=== SILVER LAYER: Creating Clean Tables ===")
print()

# Silver Patients - Clean and standardize
silver_patients = spark.table("healthcare.hospital.bronze_patients") \
    .dropDuplicates(["patient_id"]) \
    .filter(col("patient_id").isNotNull()) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("gender", upper(col("gender"))) \
    .withColumn("email", lower(col("email"))) \
    .withColumn("state", upper(col("state"))) \
    .drop("load_timestamp", "batch_id", "source_system", "record_count")

silver_patients.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_patients")
print(f"✓ Silver Patients: {silver_patients.count()} records")

# Silver Doctors
silver_doctors = spark.table("healthcare.hospital.bronze_doctors") \
    .dropDuplicates(["doctor_id"]) \
    .filter(col("doctor_id").isNotNull()) \
    .withColumn("email", lower(col("email"))) \
    .drop("load_timestamp", "batch_id", "source_system", "record_count")

silver_doctors.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_doctors")
print(f"✓ Silver Doctors: {silver_doctors.count()} records")

# Silver Departments
silver_departments = spark.table("healthcare.hospital.bronze_departments") \
    .drop("load_timestamp", "batch_id", "source_system", "record_count")

silver_departments.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_departments")
print(f"✓ Silver Departments: {silver_departments.count()} records")

# Silver Appointments - Standardize status
silver_appointments = spark.table("healthcare.hospital.bronze_appointments") \
    .dropDuplicates(["appointment_id"]) \
    .filter(col("patient_id").isNotNull() & col("doctor_id").isNotNull()) \
    .withColumn("status", trim(upper(col("status")))) \
    .drop("load_timestamp", "batch_id", "source_system", "record_count")

silver_appointments.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_appointments")
print(f"✓ Silver Appointments: {silver_appointments.count()} records")

# Silver Diagnoses, Treatments, Medications
for table in ["diagnoses", "treatments", "medications", "lab_tests"]:
    df = spark.table(f"healthcare.hospital.bronze_{table}") \
        .drop("load_timestamp", "batch_id", "source_system", "record_count")
    df.write.mode("overwrite").saveAsTable(f"healthcare.hospital.silver_{table}")
    print(f"✓ Silver {table.title()}: {df.count()} records")

# Silver Billing
silver_billing = spark.table("healthcare.hospital.bronze_billing") \
    .dropDuplicates(["bill_id"]) \
    .filter(col("total_amount") >= 0) \
    .drop("load_timestamp", "batch_id", "source_system", "record_count")

silver_billing.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_billing")
print(f"✓ Silver Billing: {silver_billing.count()} records")

print()
print("=== SILVER LAYER COMPLETE ===")

=== SILVER LAYER: Creating Clean Tables ===

✓ Silver Patients: 500 records
✓ Silver Doctors: 20 records
✓ Silver Departments: 10 records
✓ Silver Appointments: 2000 records
✓ Silver Diagnoses: 15 records
✓ Silver Treatments: 15 records
✓ Silver Medications: 15 records
✓ Silver Lab_Tests: 10 records
✓ Silver Billing: 1800 records

=== SILVER LAYER COMPLETE ===


In [0]:
# BUSINESS TRANSFORMATIONS
from pyspark.sql.functions import datediff, current_date, round as spark_round, sum as spark_sum

print("=== BUSINESS TRANSFORMATIONS ===")
print()

# Calculate Patient Age
patients_with_age = spark.table("healthcare.hospital.silver_patients") \
    .withColumn("age", 
        (datediff(current_date(), col("date_of_birth")) / 365).cast("int")
    )

patients_with_age.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_patients_enriched")
print(f"✓ Patient Age Calculated: {patients_with_age.count()} records")

# Calculate Appointment Duration (already in minutes)
appts_with_metrics = spark.table("healthcare.hospital.silver_appointments") \
    .withColumn("appointment_date_only", to_date(col("appointment_date")))

appts_with_metrics.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_appointments_enriched")
print(f"✓ Appointment Metrics Calculated: {appts_with_metrics.count()} records")

# Outstanding Bills
billing_enriched = spark.table("healthcare.hospital.silver_billing") \
    .withColumn("outstanding_amount", 
        col("patient_responsibility")
    )

billing_enriched.write.mode("overwrite").saveAsTable("healthcare.hospital.silver_billing_enriched")
print(f"✓ Billing Outstanding Calculated: {billing_enriched.count()} records")

print()
print("=== BUSINESS TRANSFORMATIONS COMPLETE ===")

=== BUSINESS TRANSFORMATIONS ===

✓ Patient Age Calculated: 500 records
✓ Appointment Metrics Calculated: 2000 records
✓ Billing Outstanding Calculated: 1800 records

=== BUSINESS TRANSFORMATIONS COMPLETE ===


In [0]:
%sql
-- GOLD LAYER - Dimension Tables
-- Create dim_patient
CREATE OR REPLACE TABLE healthcare.hospital.dim_patient AS
SELECT 
  patient_id,
  first_name,
  last_name,
  gender,
  date_of_birth,
  age,
  phone,
  email,
  address,
  city,
  state,
  zip_code,
  blood_group,
  registration_date,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_patients_enriched;

SELECT COUNT(*) as total_patients FROM healthcare.hospital.dim_patient;

total_patients
500


In [0]:
%sql
-- Create dim_doctor
CREATE OR REPLACE TABLE healthcare.hospital.dim_doctor AS
SELECT 
  doctor_id,
  first_name,
  last_name,
  specialization,
  department_id,
  phone,
  email,
  license_number,
  years_of_experience,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_doctors;

-- Create dim_department
CREATE OR REPLACE TABLE healthcare.hospital.dim_department AS
SELECT 
  department_id,
  department_name,
  description,
  location,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_departments;

-- Create dim_diagnosis
CREATE OR REPLACE TABLE healthcare.hospital.dim_diagnosis AS
SELECT 
  diagnosis_id,
  diagnosis_name,
  description,
  category,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_diagnoses;

-- Create dim_treatment
CREATE OR REPLACE TABLE healthcare.hospital.dim_treatment AS
SELECT 
  treatment_id,
  treatment_name,
  description,
  base_cost,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_treatments;

-- Create dim_medication
CREATE OR REPLACE TABLE healthcare.hospital.dim_medication AS
SELECT 
  medication_id,
  medication_name,
  description,
  dosage_form,
  unit_price,
  current_timestamp() as created_at
FROM healthcare.hospital.silver_medications;

SELECT 'Dimensions Created' as status;

status
Dimensions Created


In [0]:
%sql
-- GOLD LAYER - Fact Tables

-- Create fact_appointments
CREATE OR REPLACE TABLE healthcare.hospital.fact_appointments AS
SELECT 
  a.appointment_id,
  a.patient_id,
  a.doctor_id,
  d.department_id,
  a.appointment_date,
  a.duration_minutes,
  a.status,
  a.appointment_type,
  a.reason_for_visit
FROM healthcare.hospital.silver_appointments_enriched a
JOIN healthcare.hospital.dim_doctor d ON a.doctor_id = d.doctor_id;

-- Create fact_billing
CREATE OR REPLACE TABLE healthcare.hospital.fact_billing AS
SELECT 
  bill_id,
  patient_id,
  appointment_id,
  billing_date,
  total_amount,
  insurance_coverage,
  patient_responsibility,
  outstanding_amount,
  status
FROM healthcare.hospital.silver_billing_enriched;

SELECT 
  (SELECT COUNT(*) FROM healthcare.hospital.fact_appointments) as fact_appointments,
  (SELECT COUNT(*) FROM healthcare.hospital.fact_billing) as fact_billing;

fact_appointments,fact_billing
2000,1800


In [0]:
%sql
-- HEALTHCARE ANALYTICS - Key Performance Indicators (KPIs)

SELECT 
  'Total Patients' as metric,
  COUNT(DISTINCT patient_id) as value
FROM healthcare.hospital.dim_patient

UNION ALL

SELECT 
  'Total Doctors',
  COUNT(DISTINCT doctor_id)
FROM healthcare.hospital.dim_doctor

UNION ALL

SELECT 
  'Total Departments',
  COUNT(DISTINCT department_id)
FROM healthcare.hospital.dim_department

UNION ALL

SELECT 
  'Total Appointments',
  COUNT(*)
FROM healthcare.hospital.fact_appointments

UNION ALL

SELECT 
  'Completed Appointments',
  COUNT(*)
FROM healthcare.hospital.fact_appointments
WHERE status = 'COMPLETED'

UNION ALL

SELECT 
  'Total Hospital Revenue',
  CAST(SUM(total_amount) as BIGINT)
FROM healthcare.hospital.fact_billing

UNION ALL

SELECT 
  'Total Payments Collected',
  CAST(SUM(total_amount - outstanding_amount) as BIGINT)
FROM healthcare.hospital.fact_billing

UNION ALL

SELECT 
  'Outstanding Amount',
  CAST(SUM(outstanding_amount) as BIGINT)
FROM healthcare.hospital.fact_billing;

metric,value
Total Patients,500
Total Doctors,20
Total Departments,10
Completed Appointments,499
Total Hospital Revenue,3253834
Total Payments Collected,2205417
Outstanding Amount,1048416
Total Appointments,2000


In [0]:
%sql
-- Patient Analytics - Demographics

SELECT 
  CASE 
    WHEN age < 18 THEN 'Child (0-17)'
    WHEN age BETWEEN 18 AND 35 THEN 'Young Adult (18-35)'
    WHEN age BETWEEN 36 AND 55 THEN 'Middle Age (36-55)'
    WHEN age BETWEEN 56 AND 70 THEN 'Senior (56-70)'
    ELSE 'Elderly (70+)'
  END as age_group,
  COUNT(*) as patient_count,
  ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM healthcare.hospital.dim_patient), 2) as percentage
FROM healthcare.hospital.dim_patient
GROUP BY age_group
ORDER BY patient_count DESC;

age_group,patient_count,percentage
Middle Age (36-55),128,25.60
Young Adult (18-35),116,23.20
Elderly (70+),89,17.80
Senior (56-70),88,17.60
Child (0-17),79,15.80


In [0]:
%sql
-- Doctor Performance Analytics

SELECT 
  d.doctor_id,
  CONCAT(d.first_name, ' ', d.last_name) as doctor_name,
  d.specialization,
  dept.department_name,
  COUNT(a.appointment_id) as total_appointments,
  COUNT(CASE WHEN a.status = 'COMPLETED' THEN 1 END) as completed_appointments,
  ROUND(COUNT(CASE WHEN a.status = 'COMPLETED' THEN 1 END) * 100.0 / COUNT(a.appointment_id), 2) as completion_rate
FROM healthcare.hospital.dim_doctor d
JOIN healthcare.hospital.fact_appointments a ON d.doctor_id = a.doctor_id
JOIN healthcare.hospital.dim_department dept ON d.department_id = dept.department_id
GROUP BY d.doctor_id, d.first_name, d.last_name, d.specialization, dept.department_name
ORDER BY total_appointments DESC
LIMIT 10;

doctor_id,doctor_name,specialization,department_name,total_appointments,completed_appointments,completion_rate
11,Dr. Anna Rodriguez,Cardiologist,Cardiology,114,20,17.54
1,Dr. Sarah Johnson,Cardiologist,Cardiology,112,32,28.57
4,Dr. James Brown,Pediatrician,Pediatrics,110,31,28.18
3,Dr. Emily Williams,Orthopedic Surgeon,Orthopedics,110,30,27.27
16,Dr. Mark Jackson,Emergency Medicine,Emergency,108,24,22.22
19,Dr. Karen Martin,Dermatologist,Dermatology,106,29,27.36
12,Dr. Chris Lee,Neurologist,Neurology,105,31,29.52
2,Dr. Michael Chen,Neurologist,Neurology,104,30,28.85
15,Dr. Linda Thomas,General Physician,General Medicine,101,13,12.87
18,Dr. Paul Harris,Radiologist,Radiology,101,26,25.74


In [0]:
%sql
-- Department Performance Analytics

SELECT 
  dept.department_name,
  COUNT(DISTINCT a.appointment_id) as total_appointments,
  COUNT(DISTINCT a.patient_id) as unique_patients,
  SUM(b.total_amount) as total_revenue,
  ROUND(AVG(b.total_amount), 2) as avg_bill_amount
FROM healthcare.hospital.dim_department dept
JOIN healthcare.hospital.dim_doctor doc ON dept.department_id = doc.department_id
JOIN healthcare.hospital.fact_appointments a ON doc.doctor_id = a.doctor_id
LEFT JOIN healthcare.hospital.fact_billing b ON a.appointment_id = b.appointment_id
GROUP BY dept.department_name
ORDER BY total_revenue DESC;

department_name,total_appointments,unique_patients,total_revenue,avg_bill_amount
Cardiology,226,180,378560.51999999996,1864.83
Psychiatry,193,162,360446.99999999977,1811.29
Neurology,209,176,354094.31000000006,1853.9
Dermatology,204,168,330749.7300000001,1750.0
General Medicine,186,160,329388.9100000001,1893.04
Emergency,196,158,315121.9099999999,1800.7
Orthopedics,204,170,312861.4399999999,1719.02
Radiology,193,155,305101.4200000001,1849.1
Oncology,189,152,286034.26,1765.64
Pediatrics,200,167,281474.61,1759.22


In [0]:
%sql
-- Billing Analytics Summary

SELECT 
  status,
  COUNT(*) as bill_count,
  SUM(total_amount) as total_billed,
  SUM(insurance_coverage) as total_insurance_covered,
  SUM(patient_responsibility) as total_patient_responsibility,
  SUM(outstanding_amount) as total_outstanding,
  ROUND(AVG(total_amount), 2) as avg_bill_amount
FROM healthcare.hospital.fact_billing
GROUP BY status
ORDER BY total_billed DESC;

status,bill_count,total_billed,total_insurance_covered,total_patient_responsibility,total_outstanding,avg_bill_amount
Paid,475,856517.1499999996,569798.1000000003,286719.04999999993,286719.04999999993,1803.19
Pending,440,802221.7299999997,544053.5199999999,258168.21,258168.21,1823.23
Overdue,441,799138.2799999992,547558.4600000001,251579.82000000018,251579.82000000018,1812.1
Partially Paid,444,795956.9499999993,544007.7200000006,251949.23000000004,251949.23000000004,1792.7


In [0]:
%sql
-- DATA RECONCILIATION REPORT
-- Validate record counts across Bronze → Silver → Gold

SELECT 
  'Patients' as entity,
  (SELECT COUNT(*) FROM healthcare.hospital.bronze_patients) as bronze_count,
  (SELECT COUNT(*) FROM healthcare.hospital.silver_patients_enriched) as silver_count,
  (SELECT COUNT(*) FROM healthcare.hospital.dim_patient) as gold_count

UNION ALL

SELECT 
  'Doctors',
  (SELECT COUNT(*) FROM healthcare.hospital.bronze_doctors),
  (SELECT COUNT(*) FROM healthcare.hospital.silver_doctors),
  (SELECT COUNT(*) FROM healthcare.hospital.dim_doctor)

UNION ALL

SELECT 
  'Departments',
  (SELECT COUNT(*) FROM healthcare.hospital.bronze_departments),
  (SELECT COUNT(*) FROM healthcare.hospital.silver_departments),
  (SELECT COUNT(*) FROM healthcare.hospital.dim_department)

UNION ALL

SELECT 
  'Appointments',
  (SELECT COUNT(*) FROM healthcare.hospital.bronze_appointments),
  (SELECT COUNT(*) FROM healthcare.hospital.silver_appointments_enriched),
  (SELECT COUNT(*) FROM healthcare.hospital.fact_appointments)

UNION ALL

SELECT 
  'Billing',
  (SELECT COUNT(*) FROM healthcare.hospital.bronze_billing),
  (SELECT COUNT(*) FROM healthcare.hospital.silver_billing_enriched),
  (SELECT COUNT(*) FROM healthcare.hospital.fact_billing);

entity,bronze_count,silver_count,gold_count
Patients,500,500,500
Doctors,20,20,20
Departments,10,10,10
Appointments,2000,2000,2000
Billing,1800,1800,1800


# 🌟 STAR SCHEMA STRUCTURE - Healthcare Data Warehouse

## Architecture Overview

This healthcare data warehouse follows a **Star Schema** design with:
- **6 Dimension Tables** (descriptive attributes)
- **2 Fact Tables** (measurable events/transactions)

---

## 📊 FACT TABLES (Center of the Star)

### 1. **fact_appointments** 
**Grain**: One row per appointment

| Column | Type | Description |
|--------|------|-------------|
| appointment_id | BIGINT | Primary Key |
| patient_id | BIGINT | FK → dim_patient |
| doctor_id | BIGINT | FK → dim_doctor |
| department_id | BIGINT | FK → dim_department |
| appointment_date | TIMESTAMP | When |
| duration_minutes | BIGINT | Measure |
| status | STRING | Measure |
| appointment_type | STRING | Measure |
| reason_for_visit | STRING | Measure |

**Measures**: Appointment counts, durations, completion rates

### 2. **fact_billing**
**Grain**: One row per bill

| Column | Type | Description |
|--------|------|-------------|
| bill_id | BIGINT | Primary Key |
| patient_id | BIGINT | FK → dim_patient |
| appointment_id | BIGINT | FK → fact_appointments |
| billing_date | TIMESTAMP | When |
| total_amount | DOUBLE | Measure |
| insurance_coverage | DOUBLE | Measure |
| patient_responsibility | DOUBLE | Measure |
| outstanding_amount | DOUBLE | Measure |
| status | STRING | Measure |

**Measures**: Revenue, payments, outstanding amounts, insurance coverage

---

## ⭐ DIMENSION TABLES (Points of the Star)

### 1. **dim_patient** (Patient Demographics)
- patient_id (PK)
- first_name, last_name
- gender, date_of_birth, **age** (calculated)
- contact info (phone, email, address)
- blood_group, registration_date

### 2. **dim_doctor** (Physician Information)
- doctor_id (PK)
- first_name, last_name
- specialization
- department_id
- license_number, years_of_experience
- contact info

### 3. **dim_department** (Hospital Departments)
- department_id (PK)
- department_name
- description, location

### 4. **dim_diagnosis** (Diagnosis Catalog)
- diagnosis_id (PK)
- diagnosis_name
- description, category

### 5. **dim_treatment** (Treatment Catalog)
- treatment_id (PK)
- treatment_name
- description, base_cost

### 6. **dim_medication** (Medication Catalog)
- medication_id (PK)
- medication_name
- description, dosage_form, unit_price

---

## 🔗 STAR SCHEMA RELATIONSHIPS

```
                    dim_patient
                         |
                         | patient_id
                         |
                         ↓
    dim_doctor ──→ fact_appointments ←── dim_department
                         |
                         | appointment_id
                         |
                         ↓
    dim_patient ──→ fact_billing
```

---

## 📈 ANALYTICAL CAPABILITIES

### Enabled Queries:
1. **Patient Analytics**: Demographics, age groups, visit patterns
2. **Doctor Performance**: Appointment loads, completion rates, specialization analysis
3. **Department Performance**: Revenue by department, patient volume, resource utilization
4. **Financial Analytics**: Revenue, outstanding balances, insurance coverage, payment trends
5. **Operational Metrics**: Appointment status, wait times, capacity utilization

### Query Example:
```sql
SELECT 
  d.department_name,
  COUNT(DISTINCT fa.patient_id) as unique_patients,
  COUNT(fa.appointment_id) as total_appointments,
  SUM(fb.total_amount) as total_revenue
FROM fact_appointments fa
JOIN dim_department d ON fa.department_id = d.department_id
JOIN fact_billing fb ON fa.appointment_id = fb.appointment_id
GROUP BY d.department_name;
```

---

## ✅ DATA QUALITY
- **Bronze Layer**: Raw data with metadata
- **Silver Layer**: Cleaned, deduplicated, standardized
- **Gold Layer**: Business-ready star schema
- **Reconciliation**: Validated across all layers